In [ ]:
from pathlib import Path
import re

import pandas as pd
from lxml import etree


In [ ]:
data_dir = Path('../data/xml')
xml_files = sorted(data_dir.glob('*.xml'))
len(xml_files), [p.name for p in xml_files]

In [ ]:
parser = etree.XMLParser(recover=True, huge_tree=True)

test_doc = etree.parse(str(xml_files[0]), parser)
root = test_doc.getroot()

root.tag, root.nsmap

In [ ]:
def get_tei_ns(root) -> dict:
    """Gibt einen Namespace-Mapper für TEI zurück (falls Default-Namespace gesetzt ist)."""
    ns = root.nsmap.get(None)
    return {'tei': ns} if ns else {}

ns = get_tei_ns(root)
ns

In [ ]:
letter_divs = root.xpath(".//tei:div[@subtype='letter']", namespaces=ns) if ns else root.xpath(".//*[@subtype='letter' and local-name()='div']")
len(letter_divs)

In [ ]:
one = letter_divs[1]  # --> wir schauen uns den zweiten Brief an

# Briefnummer
one.get('n')

In [ ]:
# Buchdiv als Vorfahr suchen
book_div = one.xpath("ancestor::tei:div[@subtype='Book'][1]", namespaces=ns) if ns else one.xpath("ancestor::*[local-name()='div' and @subtype='Book'][1]")
book_n = book_div[0].get('n') if book_div else None
book_n

In [ ]:
def norm_ws(s: str) -> str:
    return re.sub(r'\s+', ' ', s).strip()

def text_of_first(nodes):
    if not nodes:
        return None
    return norm_ws(' '.join(nodes[0].itertext()))

dateline = text_of_first(one.xpath(".//tei:seg[@rend='dateline']", namespaces=ns)) if ns else text_of_first(one.xpath(".//*[local-name()='seg' and @rend='dateline']"))
salute = text_of_first(one.xpath(".//tei:seg[@rend='salute']", namespaces=ns)) if ns else text_of_first(one.xpath(".//*[local-name()='seg' and @rend='salute']"))
date_el = one.xpath(".//tei:date[1]", namespaces=ns) if ns else one.xpath(".//*[local-name()='date'][1]")
date_when = date_el[0].get('when') if date_el else None

dateline, salute, date_when

In [ ]:
def extract_sender_recipient_from_salute(salute):
    """Heuristik für:
    - 'Cicero Bruto salutem' -> ('Cicero', 'Bruto')
    - 'M. Cicero S.D. P. Lentulo procos.' -> ('M. Cicero', 'P. Lentulo procos')
    """
    if not salute:
        return None, None

    s = norm_ws(str(salute)).replace('"', '').replace("'", '')
    # Schlusswort salutem am Ende entfernen
    s = re.sub(r'\b(salutem)\b\.?\s*$', '', s, flags=re.IGNORECASE).strip()

    # Trenner S.D. (salutem dicit)
    m = re.search(r'\bS\s*\.?\s*D\s*\.?\b', s, flags=re.IGNORECASE)
    if m:
        sender = norm_ws(s[:m.start()]) or None
        recipient = norm_ws(s[m.end():])
        recipient = re.sub(r'^[\s\.,;:]+', '', recipient)
        recipient = re.sub(r'[\s\.,;:]+$', '', recipient)
        recipient = recipient or None
        return sender, recipient

    parts = s.split()
    if len(parts) < 2:
        return None, None
    return parts[0], parts[1]

extract_sender_recipient_from_salute(salute)

In [ ]:
def parse_year(date_when):
    if not date_when:
        return None
    s = str(date_when).strip()
    return int(s) if re.match(r'^-?\d{1,4}$', s) else None

print(parse_year(date_when))

In [ ]:
sender, recipient = extract_sender_recipient_from_salute(salute)

row = {
    'corpus': 'TEST',
    'book_n': book_n,
    'letter_n': one.get('n'),
    'date_when': date_when,
    'year': parse_year(date_when),
    'dateline': dateline,
    'salute': salute,
    'sender': sender,
    'recipient': recipient,
}

row

In [ ]:
def corpus_label_from_filename(p: Path) -> str:
    name = p.name.lower()
    if 'ad-familiares' in name:
        return 'ad_familiares'
    if 'ad-atticum' in name:
        return 'ad_atticum'
    if 'ad-quintum' in name:
        return 'ad_quintum_fratrem'
    if 'ad-brutum' in name:
        return 'ad_m_brutum'
    return p.stem

rows = []

for path in xml_files:
    corpus = corpus_label_from_filename(path)
    doc = etree.parse(str(path), parser)
    root = doc.getroot()
    ns = get_tei_ns(root)

    letter_divs = root.xpath(".//tei:div[@subtype='letter']", namespaces=ns) if ns else root.xpath(".//*[@subtype='letter' and local-name()='div']")

    for ld in letter_divs:
        book_div = ld.xpath("ancestor::tei:div[@subtype='Book'][1]", namespaces=ns) if ns else ld.xpath("ancestor::*[local-name()='div' and @subtype='Book'][1]")
        book_n = book_div[0].get('n') if book_div else None
        letter_n = ld.get('n')

        dateline = text_of_first(ld.xpath(".//tei:seg[@rend='dateline']", namespaces=ns)) if ns else text_of_first(ld.xpath(".//*[local-name()='seg' and @rend='dateline']"))
        salute = text_of_first(ld.xpath(".//tei:seg[@rend='salute']", namespaces=ns)) if ns else text_of_first(ld.xpath(".//*[local-name()='seg' and @rend='salute']"))
        date_el = ld.xpath(".//tei:date[1]", namespaces=ns) if ns else ld.xpath(".//*[local-name()='date'][1]")
        date_when = date_el[0].get('when') if date_el else None

        sender, recipient = extract_sender_recipient_from_salute(salute)

        #für text:
        paragraphs = ld.xpath(".//tei:p", namespaces=ns) if ns else ld.xpath(".//*[local-name()='p']")
        text = " ".join(" ".join(t.strip() for t in p.itertext() if t and t.strip()) for p in paragraphs)
        text = re.sub(r"\s+", " ", text).strip()

        rows.append({
            'corpus': corpus,
            'book_n': book_n,
            'letter_n': letter_n,
            'date_when': date_when,
            'year': parse_year(date_when),
            'dateline': dateline,
            'salute': salute,
            'sender': sender,
            'recipient': recipient,
            'text': text
        })

len(rows)

In [ ]:
df = pd.DataFrame(rows)


In [ ]:
probleme = df[df['recipient'].isna()][['salute', 'sender']]

In [ ]:
def korrekturen_anwenden(row):
    if row['salute'] in korrekturen:
        row['sender'], row['recipient'] = korrekturen[row['salute']]
    return row

In [ ]:
korrekturen = {
    "CICERO APPIO IMP. S. D.": ("CICERO", "APPIO IMP."),
    "CICERO APPIO PVLCHRO VT SPERO, CENSORI S. D.": ("CICERO", "APPIO PVLCHRO"),
    "M. TVLLIVS M. F. CICERO Q. METELLO Q. F. CELERI PROCOS. S. D.": ("CICERO", "Q. METELLO Q.F.CELERI"),
    "CICERO CAESARI IMP. S. D.":("CICERO", "CAESARI IMP."),
    "CICERO CVRIO S. D." : ("CICERO","CVRIO"),
    "CICERO PAETO S. D." : ("CICERO", "PAETO"),
    "C. ASINIVS POLLIO CICERONI S. D.": ("C. ASINIVS POLLIO", "CICERONI"),
    "D. BRVTVS COS. DESIG. M. CICERONI S. D.": ("D. BRVTVS COS. DESIG.", "M.CICERONI"),
    "M. CICERO D. BRVTO COS. DESIG. S. D.": ("M. CICERO", "D. BRVTO COS. DESIG."),
    "M. CICERO D. BRVTO COS. DES. S. D.": ("M. CICERO", "D. BRVTO COS. DES."),
    "M. CICERO D. BRVTO S. D.": ("M. CICERO", "D. BRVTO"),
    "D. BRVTO COS. DESIG." : ("CICERO", "OPPIO"),
    "CICERO C. SEXTILIO RVFO QVAESTORI S. D.": ("CICERO", "C. SEXTILIO RVFO QVAESTORI" ),
    "CICERO P. CAESIO S. D.": ("CICERO","P.CAESIO"),
    "M. CICERO T. TITIO T. F. LEG. S. D.": ("M. CICERO", "T. TITIO T. F. LEG."),
    "M. CICERO IIII VIRIS ET DECVRIONIBVS S. D." : ("M. CICERO", "IIII VIRIS ET DECVRIONIBVS"),
    "TVLLIVS TERENTIAE SVAE, TVLLIOLAE SVAE, CICERONI SVO S. D.": ("TVLLIVS","TERENTIAE SVAE, TVLLIOLAE SVAE, CICERONI SVO"),
    "TVLLIVS TERENTIAE SVAE S. D.": ("TVLLIVS", "TERENTIAE SVAE"),
    "Q. CICERO TIRONI S. D." : ("Q.CICERO", "TIRONI"),
    "QVINTVS TIRONI SV0 P. S. D.": ("QVINTVS", "TIRONI SVO P."),
    "CICERO ATTICO S. D.": ("CICERO", "ATTICO"),
    "CICERO ANTONIO COS. S. D.": ("CICERO", "ANTONIO COS.")
    }

In [ ]:

df = df.apply(korrekturen_anwenden, axis=1)

In [ ]:
def Namenssplitting_besser(row):
    if pd.notna(row['sender']) and pd.notna(row['recipient']):
        if re.fullmatch(r'[A-Z]\.', row['sender']) and row['recipient'] == 'CICERO':
            parts = row['salute'].split("CICERO", 1)
            if len(parts) == 2:
                row['sender'] = parts[0].strip() + " CICERO"
                row['recipient'] = parts[1].strip()
                row['recipient'] = re.sub(r'\bS\..*$', '', row['recipient']).strip()
    return row

In [ ]:

df = df.apply(Namenssplitting_besser, axis=1)

In [ ]:
def fix_teremia(row):
    if pd.notna(row['sender']):
        if row['sender'] == 'TVLLIVS TEREMIAE SVAE':

            row['sender'] = 'TVLLIVS'
            row['recipient'] = 'TEREMIAE'

    return row

In [ ]:

df = df.apply(fix_teremia, axis=1)

In [ ]:
def mehrere_absender(row):
    if pd.notna(row['sender']):
        if row['sender'] == 'TVLLIVS ET CICERO, TERENTIA, TVLLIA Q. Q. TIRONI S. P. D.':

            row['sender'] = 'TVLLIVS ET CICERO, TERENTIA, TVLLIA'
            row['recipient'] = 'TIRONI'

    return row

In [ ]:
df = df.apply(mehrere_absender, axis=1)

In [ ]:
def Daten_besser_matchen(row):
    form = r'\((\d+)\)'
    match = re.search(form, row['dateline'])
    if pd.isna(row['date_when']):
        if match:
            row['date_when'] = match.group(1) 
    return row

In [ ]:
df = df.apply(Daten_besser_matchen, axis=1)


In [ ]:
def recipients_aufräumen(row):
    if pd.notna(row['recipient']):
        if row['recipient'] == 'TERENTIAE SVAE' or row['recipient'] == 'TERENTIAE SVAE' or row['recipient'] == 'TEREMIA' or row['recipient'] == 'TERENTIAE 5VAE' or row['recipient'] == 'TEREMIAE' or row['recipient'] == 'TERENTIAE':
            row['recipient'] = 'TERENTIA'
        elif row['recipient'] == 'TERENTIAE ET TVLLIAE ET CICERONI SVIS' or row['recipient'] == 'TERENTIAE SVAE ET TVLLIAE ET CICERONI' or row['recipient']== 'TERENTIAE SVAE ET TVLLIOLAE ET CICERONI SVIS' or row['recipient'] == 'TERENTIAE SVAE, TVLLIOLAE SVAE, CICERONI SVO':
            row['recipient'] = 'Familie'
        elif row['recipient'] == 'TIRONI SVO':
            row['recipient'] = 'TIRONI'


    return row

In [ ]:
df = df.apply(recipients_aufräumen, axis=1)

In [ ]:
def pulchro_richtig(text):
    endungen = [(r'CHRO\b', 'CHER')]
    if pd.notna(text):
        for muster, ersatz in endungen:
            text = re.sub(muster, ersatz, text)
        return text

In [ ]:
df['recipient'] = df['recipient'].apply(pulchro_richtig)

In [ ]:
def dativ_entfernen(text):
    endungen = [(r'AE\b', 'A'), (r'O\b', 'VS'),(r'ONI\b', 'O') ]
    if pd.notna(text):
        for muster, ersatz in endungen:
            text = re.sub(muster, ersatz, text)
        return text

In [ ]:
df['recipient'] = df['recipient'].apply(dativ_entfernen)

In [ ]:
def jahreszahlen_aufbereiten(row):
    if pd.isna(row['date_when']):
        return row
    if re.search(r'-00\d\d', row['date_when']):
        row['year'] = int(row['date_when'])
    else:
        row['year'] = -int(row['date_when'])
    return row
    


In [ ]:
df = df.apply(jahreszahlen_aufbereiten, axis=1)


In [ ]:
names_recode = [

    # Quintus Tullius Cicero zuerst, damit Cicero nicht damit matcht

    (r"Q\.?\s*CICERO", "QUINTUS TULLIUS CICERO"),
    (r"\bQVINTVS", "QUINTUS TULLIUS CICERO (wahrscheinlich)"),


    # Marcus Tullius Cicero, um möglichst alle Ciceros abzufangen. Hinweis: Tullius Terentiae suae etc. wird hier 
    # auch abgefangen, weil der ganze Teil mit der Familie Teil der recipients wäre (sieht man am grammatikalischen Fall), aber
    # Cicero selbst schreibt (da Nominativ)

    (r"\bTVLLIVS\b.*\bCICERO\b", "MARCUS TULLIUS CICERO"),
    (r"M\.?\s*TVLLIVS\s*M\.\s*F\.\s*CICERO\s*PROCOS\.", "MARCUS TULLIUS CICERO"),
    (r"M\.?\s*TVLLIVS\s*M\.\s*F\.\s*M\.\s*N\.\s*CICERO\s*IMP\.", "MARCUS TULLIUS CICERO"),
    (r"\bTVLLIVS\b.*\bCICERO\b", "MARCUS TULLIUS CICERO"),
    (r"M\.?\s*TVLLIVS\s*M\.\s*F\.\s*CICERO\s*PROCOS\.", "MARCUS TULLIUS CICERO"),
    (r"M\.?\s*TVLLIVS\s*M\.\s*F\.\s*M\.\s*N\.\s*CICERO\s*IMP\.", "MARCUS TULLIUS CICERO"),
    (r"M\.?\s*CICERO", "MARCUS TULLIUS CICERO"),
    (r"\bTVLLIVS\b", "MARCUS TULLIUS CICERO"),
    (r"\bMARCVS\b", "MARCUS TULLIUS CICERO"),
    (r"CICERO", "MARCUS TULLIUS CICERO"),

    # CAELIUS, auch mit 5, da OCR

    (r"CAELIV[S5]", "MARCUS CAELIUS RUFUS"),


    # Plancus

    (r"PLANCVS\s*IMR\s*COS\.\s*DESIG\.", "LUCIUS MUNATIUS PLANCUS"),
    (r"PLANCVS\s*IMR\s*COS\.\s*DES\.", "LUCIUS MUNATIUS PLANCUS"),
    (r"\bPLANCVS\b", "LUCIUS MUNATIUS PLANCUS"),

    # Die beiden Bruti (Marcus Iunius Brutus und Decimus Brutus, sowie Brutus mit Cassius)

    # Decimus Brutus

    (r"D\.?\s*BRVTVS\s*COS\.\s*DESIG\.", "DECIMUS IUNIUS BRUTUS ALBINUS"),
    (r"D\.?\s*BRVTVS\s*IMR\s*COS\.\s*DES\.", "DECIMUS IUNIUS BRUTUS ALBINUS"),
    (r"D\.?\s*BRVTVS\s*IMP\.\s*COS\.\s*DESIG\.", "DECIMUS IUNIUS BRUTUS ALBINUS"),
    (r"D\.?\s*BRVTVS", "DECIMUS IUNIUS BRUTUS ALBINUS (wahrscheinlich)"),

    # Brutus und Cassius

    (r"\bBRVTVS\b.*\bCASSIVS\b", "MARCUS IUNIUS BRUTUS und GAIUS CASSIUS LONGINUS (Gruppe)"),

    # Marcus Iunius Brutus

    (r"\bBRVTVS\b", "MARCUS IUNIUS BRUTUS"),
    (r"\bBRUTUS\b", "MARCUS IUNIUS BRUTUS"),

    # Caesar (2 Varianten)

    (r"CAESAR\s*IMP\.", "GAIUS IULIUS CAESAR"),
    (r"\bCAESAR\b", "GAIUS IULIUS CAESAR"),

    # Pompeius

    (r"CN\.\s*MAGNVS\s*PROCOS\.", "GNAEUS POMPEIUS MAGNUS"),

    # Lepidus

    (r"M\.\s*LEPIDVS\s*IMP\.\s*ITER\.\s*PONT\.\s*MAX\.", "MARCUS AEMILIUS LEPIDUS"),
    (r"LEPIDVS\s*IMP\.\s*ITER\.\s*PONL\s*MAX\.", "MARCUS AEMILIUS LEPIDUS"),
    (r"\bLEPIDVS\b", "MARCUS AEMILIUS LEPIDUS"),

    # Gaius Asinius Pollio

    (r"C\.\s*ASINIVS\s*POLLIO", "GAIUS ASINIUS POLLIO"),
    (r"\bPOLLIO\b", "GAIUS ASINIUS POLLIO"),

    # Cassius

    (r"C\.\s*CASSIVS\s*PROCOS\.", "GAIUS CASSIUS LONGINUS"),
    (r"C\.\s*CASSIVS\s*Q\.", "GAIUS CASSIUS LONGINUS"),
    (r"C\.\s*CASSIVS", "GAIUS CASSIUS LONGINUS"),
    (r"\bCASSIVS\b", "GAIUS CASSIUS LONGINUS"),

    # Balbus und Oppius

    (r"BALBVS\s*ET\s*OPPIVS", "LUCIUS CORNELIUS BALBUS (Maior) und GAIUS OPPIUS"),


    # Balbus

    (r"\bBALBVS\b", "LUCIUS CORNELIUS BALBUS"),

    # Dolabella

    (r"\bDOLABELLA\b", "PUBLIUS CORNELIUS DOLABELLA"),

    # Bithynicus

    (r"\bBITHYNICVS", "QUINTUS POMPEIUS BITHYNICUS"),

    # Trebonius

    (r"\bTREBONIVS\b", "GAIUS TREBONIUS"),

    # Lentulus

    (r"P\.\s*LENTVLVS\s*P\.\s*F\.\s*PROQ\.\s*PROPR\.", "PUBLIUS CORNELIUS LENTULUS SURA"),
    (r"\bLENTVLVS\b", "PUBLIUS CORNELIUS LENTULUS SURA"),

    # Cato

    (r"M\.\s*CATO", "MARCUS PORTIUS CATO (Minor)"),


    # Servius

    (r"\bSERVIVS\b", "SERVIUS SULPICIUS RUFUS"),


    # Vatinius

    (r"\bVATINIVS\b", "PUBLIUS VATINIUS"),

    # Metellus Celer

    (r"Q\.\s*METELLVS\s*Q\.\s*F\.\s*CELER\s*PROCOS\.", "QUINTUS CAECILIUS METELLUS CELER"),

    # Metellus Nepos

    (r"Q\.\s*METELLVS\s*NEPOS", "QUINTUS CAECILIUS METELLUS NEPOS"),


    # Lucius Lucceius

    (r"L\.\s*LVCCEIVS\s*Q\.\s*F\.", "LUCIUS LUCCEIUS"),


    # Marcellus (wahrscheinlich)

    (r"\bMARCELLVS\b", "MARCUS CLAUDIUS MARCELLUS (wahrscheinlich)"),

    # Caecina

    (r"\bCAECINA\b", "AULUS CAECINA"),

    # Curius

    (r"\bCVRIVS\b", "QUINTUS CURIUS (wahrscheinlich)"),


    # Antonius

    (r"ANTONIVS\s*COS\.", "MARCUS ANTONIUS"),
    (r"\bANTONIVS\b", "MARCUS ANTONIUS"),

    # Galba

    (r"\bGALBA\b", "SERVIUS SULPICIUS GALBA"),

    # Matius

    (r"\bMATIVS\b", "GAIUS MATIUS"), 


    # Unklare (z.B. ABKÜRZUNGEN)

    (r"\bC\.\s*", "UNKLAR (C.)"),
    (r"\bD\.\s*", "UNKLAR (D.)"),
    (r"\bM\.\s*", "UNKLAR (M.)"),
    (r"\bSVIS\b", "UNKLAR (SVIS)"),

]


In [ ]:
def clean_sender(sender): 
    if pd.isna(sender):
        return sender

    s = sender.upper().strip()

    for pattern, new_name in names_recode:
        if re.search(pattern, s):
            return new_name

    return s

In [ ]:
df["sender_c"] = df["sender"].apply(clean_sender)

In [ ]:
out_dir = Path('../data')
out_dir.mkdir(exist_ok=True)

mini_cols = ['corpus','book_n','letter_n','sender','sender_c','recipient','date_when','year','dateline', 'text']
mini = df[mini_cols].copy()

mini_path = out_dir / 'cicero_letters.csv'
mini.to_csv(mini_path, index=False)
mini_path

In [ ]:

print(df["sender"].value_counts().to_string())
print(df["sender_c"].value_counts().to_string())